In [20]:
import numpy as np          
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.datasets import fetch_openml 
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
#importing the MNIST dataset
#naming the data as X for the images, and y for the label

mnist = fetch_openml('mnist_784', version=1)
X = mnist.data
y = mnist.target

In [3]:
X_train = X[:60000]
y_train = y[:60000]
X_test = X[60000:]
y_test = y[60000:]

In [4]:
scaler = MinMaxScaler()
norm_x_train = scaler.fit_transform(X_train)
norm_x_test = scaler.transform(X_test)

## Voting Classifier

In [5]:
clf_tree = DecisionTreeClassifier(max_depth=15) 
clf_knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
clf_log = LogisticRegression(C=100.0, max_iter=1000, n_jobs=-1)

ensemble_model = VotingClassifier(
    estimators=[
        ('Decision Tree', clf_tree), 
        ('KNN', clf_knn), 
        ('Logistic Regression', clf_log)
    ],
    voting='soft'
)

In [6]:
ensemble_model.fit(norm_x_train, y_train)

ensemble_score = ensemble_model.score(norm_x_test, y_test)

print(f" (Ensemble Score): {ensemble_score * 100:.2f}%")

 (Ensemble Score): 96.00%


## Bagging

### Decision Tree

In [7]:
base_tree = DecisionTreeClassifier(max_depth=15)

In [8]:
bagging_model = BaggingClassifier(
    estimator=base_tree, 
    n_estimators=50,    
    n_jobs=-1,            
    random_state=42      
)

In [9]:
bagging_model.fit(X_train, y_train) 

bagging_score = bagging_model.score(X_test, y_test)

print(f" Bagging accuracy: {bagging_score * 100:.2f}%")

 Bagging accuracy: 95.31%


## RandomForest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=50,      
    max_depth=15,        
    n_jobs=-1,          
    random_state=42      
)

In [11]:
rf_model.fit(X_train, y_train)

rf_score = rf_model.score(X_test, y_test)

print(f"RandomForest accuracy: {rf_score * 100:.2f}%")

RandomForest accuracy: 96.21%


### Trying to improve the model

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=1000,      
    max_depth= None,        
    n_jobs=-1,          
    random_state=42      
)

In [13]:
rf_model.fit(X_train, y_train)

rf_score = rf_model.score(X_test, y_test)

print(f"RandomForest accuracy: {rf_score * 100:.2f}%")

RandomForest accuracy: 97.25%


## Stacking

In [14]:
level0 = [
    ('knn', KNeighborsClassifier(n_neighbors=3, weights='distance')),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42))
]

level1 = LogisticRegression(max_iter=1000)

In [15]:
stacking_model = StackingClassifier(
    estimators=level0,
    final_estimator=level1,
    cv=3,        
    n_jobs=1     
)

In [16]:
stacking_model.fit(norm_x_train, y_train)

stacking_score = stacking_model.score(norm_x_test, y_test)

print(f"Stacking Accuracy: {stacking_score * 100:.2f}%")

Stacking Accuracy: 97.84%


## AdaBoost

In [17]:
base_tree = DecisionTreeClassifier(max_depth=5) 

In [18]:
ada_model = AdaBoostClassifier(
    estimator=base_tree,  
    n_estimators=100,      
    learning_rate=0.5,     
    random_state=42
)


In [19]:
ada_model.fit(X_train, y_train)

ada_score = ada_model.score(X_test, y_test)

print(f"AdaBoost Accuracy: {ada_score * 100:.2f}%")

/opt/conda/envs/learn-ai-1/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy: 91.83%


## Gradient Boosting

In [21]:
gb_model = HistGradientBoostingClassifier(
    max_iter=100,       
    learning_rate=0.1,    
    max_depth=15,        
    random_state=42
)

In [23]:
gb_model.fit(X_train, y_train)

gb_score = gb_model.score(X_test, y_test)

print(f"Gradient Boosting Accuracy: {gb_score * 100:.2f}%")

Gradient Boosting Accuracy: 97.84%
